In [ ]:
# Imports
import sys
import logging
from datetime import datetime
import pandas as pd
from IPython.display import display

sys.path.insert(0, '../../../LOGOS')
from src import Pert, plot_gantt_chart, plot_resource_utilization, plot_location_utilization, plot_equipment_utilization
# Configure logging in the runner (avoid setting basicConfig inside the module)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
import json
from pathlib import Path

cwd = Path.cwd()
benchmark = cwd/'benchmarks'
benchmark_results_file = benchmark/'priority_rules_results.json'

## Load Benchmark results
with open(benchmark_results_file, "r", encoding="utf-8") as f:
    benchmark_data = json.load(f)

In [ ]:
def run_case(case_name, file_name, json_path,
             schema_file="outage_schema.json",
             benchmark_data=None):
    """
    Runs scheduling comparison for a given case and file.

    Parameters:
        case_name (str): Case identifier (e.g., 'j60')
        file_name (str): File name (e.g., 'j601_1.sm')
        json_path (str): Path to JSON file for Pert model
        schema_file (str): Path to schema file (default: outage_schema.json)
        benchmark_data (dict): Benchmark dataset for RCPSP comparison

    Returns:
        results_df (pd.DataFrame): LOGOS.CPM results
        data_df (pd.DataFrame): RCPSP benchmark results
    """

    results_sgs = {}
    results_pgs = {}
    results_pgs_pr = {}


    # Load Pert Model
    pert = Pert.from_json_file(json_path, schema_path=schema_file)

    prs = [
        'es','ef','ls','lf', 'duration','random',
        'mts', 'mtp', 'grpw', 'grd', 'rr', 'avgrr',
        'maxrr', 'minrr', 'irsm','wcs','acs',
        'mehh_8000_b','mehh_3375_b',
        'mehh_1000_b','mehh_125_b','gphh_b'
    ]

    # Compute results with Serial
    for rule in prs:
        out = pert.calculateSerialScheduleWithResources(priority_rule=rule)
        results_sgs[rule] = out['scheduled_duration'] - 2  # remove start/end duration
        violations, is_feasible = pert.check_dependency_violations()
        if not is_feasible:
            print(violations)
    sgs = ['first', 'max_use_res_ranked', 'max_use_res_shuffled', 'md_knapsack', 'look_ahead']

    # Compute results with Parallel
    for s in sgs:
        out = pert.calculateScheduleWithResources(sgs=s)
        results_pgs[s] = out['scheduled_duration'] - 2  # remove start/end duration
        violations, is_feasible = pert.check_dependency_violations()
        if not is_feasible:
            print(violations)

    for s in sgs:
        results_pgs_pr[s] = {}
        for rule in prs:
            out = pert.calculateScheduleWithResources(sgs=s, priority_rule=rule)
            results_pgs_pr[s][rule] = out['scheduled_duration'] - 2  # remove start/end duration
            violations, is_feasible = pert.check_dependency_violations()
            if not is_feasible:
                print(violations)



    print('Results from LOGOS.CPM Using Serial Generation Scheme:')
    print('-' * 60)
    results_df_sgs = pd.DataFrame(results_sgs, index=[0])
    display(results_df_sgs)

    if benchmark_data is None:
        raise ValueError("benchmark_data must be provided")

    data = benchmark_data[case_name][file_name]
    data_df = pd.DataFrame(data, index=[0]).filter(like='serial_forward')
    data_df.columns = data_df.columns.str.replace("_serial_forward", "", regex=False)
    data_df.columns = data_df.columns.str.lower()

    display(data_df)

    print('Results from LOGOS.CPM Using Parallel Generation Scheme:')
    print('-' * 60)
    results_df_pgs = pd.DataFrame(results_pgs, index=[0])
    display(results_df_pgs)

    for s in sgs:
        print(f'Results from LOGOS.CPM Using Parallel Generation Scheme "{s}" with Priority Rule:')
        print('-' * 60)
        results_df_pgs_pr = pd.DataFrame(results_pgs_pr[s], index=[0])
        display(results_df_pgs_pr)

    # RCPSP benchmark comparison
    print('Results from RCPSP')
    print('-' * 60)

    data_df = pd.DataFrame(data, index=[0]).filter(like='parallel_forward')
    data_df.columns = data_df.columns.str.replace("_parallel_forward", "", regex=False)
    data_df.columns = data_df.columns.str.lower()

    display(data_df)


    return results_df_sgs, results_df_pgs, data_df

## Scheduling with 30 activities

In [ ]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j30',
    file_name='j301_1.sm',
    json_path='j301_1.json',
    benchmark_data=benchmark_data
)


## Scheduling with 60 activities

In [ ]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j60',
    file_name='j601_1.sm',
    json_path='j601_1.json',
    benchmark_data=benchmark_data
)

## Scheduling with 90 activities

In [ ]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j90',
    file_name='j901_1.sm',
    json_path='j901_1.json',
    benchmark_data=benchmark_data
)

## Scheduling with 120 activities

In [ ]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j120',
    file_name='j1201_1.sm',
    json_path='j1201_1.json',
    benchmark_data=benchmark_data
)